In [1]:
# 1. Connexion au Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. On s'assure que les bibliothèques nécessaires sont là
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import pandas as pd
from tqdm import tqdm # Pour avoir une barre de progression

Mounted at /content/drive


In [2]:
# --- CHOIX DE L'ANIMAL ---
# Change cette valeur manuellement avant de lancer le code
ANIMAL_TO_PROCESS = "HiP417" # Options: "HiPsh435", "HiP659", "HiP417"

print(f"L'extraction ne traitera que les fichiers de l'animal : {ANIMAL_TO_PROCESS}")

L'extraction ne traitera que les fichiers de l'animal : HiP417


In [3]:
# 1. Processeur
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Chargement DINOv2 Large
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').to(device)

# Activation du mode "Vitesse maximale" si GPU présent
if device.type == 'cuda':
    model = model.half()
    print("Mode demi-précision (FP16) activé.")

model.eval()

import numpy as np

# 3. Transformation optimisée pour DINOv2 (Multiple de 14 et arrondis au plus proche)
def get_dinov2_size(width, height=518):
    # On force la hauteur à 518 (déjà multiple de 14)
    # On arrondit la largeur au multiple de 14 le plus proche
    new_w = int(round(width / 14) * 14)
    return [height, new_w]

transform = T.Compose([
    # Lambda qui ajuste la taille dynamiquement par image
    T.Lambda(lambda img: T.functional.resize(img, get_dinov2_size(img.size[0], img.size[1]))),
    T.ToTensor(),
    # Normalisation standard pour les modèles Vision Transformer (DINO/ViT)
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # A CONSERVER !! + 14 % d'accuracy
    ])

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:09<00:00, 127MB/s]


Mode demi-précision (FP16) activé.


In [4]:
# --- CONFIGURATION DES CHEMINS ---
# path vers les spectros zippés
path_zip = "/content/drive/MyDrive/audio_classification_sam/data/data_for_prediction/3s_prediction_mel_spectrograms.zip"

# C'est ici qu'on définit l'endroit où on va extraire (le SSD local de Colab)
path_imgs = "/content/spectros_local/3s_prediction_mel_spectrograms/3s_prediction_mel_spectrograms"

# path vers le csv résulat
path_csv_final = "/content/drive/MyDrive/audio_classification_sam/embeddings/3s_prediction_mel_spectrograms_HiP417.csv"

# --- EXÉCUTION ---
# 1. Création du dossier local
!mkdir -p {path_imgs}

# 2. On lance l'unzip
print(f"Début de l'extraction de : {path_zip}")

!unzip -q "{path_zip}" -d {path_imgs}

print("Extraction terminée")

Début de l'extraction de : /content/drive/MyDrive/audio_classification_sam/data/data_for_prediction/3s_prediction_mel_spectrograms.zip
error:  zipfile read error
Extraction terminée


In [6]:
# --- CELLULE DE NETTOYAGE DES DIMENSIONS (Version Anti-Bug) ---
import os
from PIL import Image
from collections import Counter
from tqdm import tqdm

target_dir = path_imgs
print(f"Analyse des fichiers dans : {target_dir}")

# Gestion automatique du sous-dossier
if os.path.exists(target_dir):
    content = [f for f in os.listdir(target_dir) if not f.startswith('.')]
    if len(content) == 1 and os.path.isdir(os.path.join(target_dir, content[0])):
        target_dir = os.path.join(target_dir, content[0])
        print(f"Utilisation du sous-dossier detecte : {target_dir}")

all_files = [f for f in os.listdir(target_dir) if f.endswith('.png')]
widths = []
corrupted_files = []

print("Scan des dimensions et intégrité...")
for f in tqdm(all_files):
    file_path = os.path.join(target_dir, f)
    try:
        with Image.open(file_path) as img:
            widths.append(img.size[0])
    except Exception:
        # Si le fichier est illisible, on le note pour suppression
        corrupted_files.append(f)

# 1. Traitement des fichiers corrompus
if corrupted_files:
    print(f"\n⚠️ {len(corrupted_files)} fichier(s) corrompu(s) détecté(s). Suppression...")
    for f in corrupted_files:
        os.remove(os.path.join(target_dir, f))
    # On met à jour la liste des fichiers valides pour la suite
    all_files = [f for f in all_files if f not in corrupted_files]

# 2. Identification de la largeur majoritaire
if not widths:
    raise ValueError("Aucune image valide trouvée.")

data_counts = Counter(widths)
main_width = data_counts.most_common(1)[0][0]
print(f"\nLargeur standard identifiée : {main_width}px")

# 3. Suppression des images de taille incorrecte
deleted_count = 0
for f in all_files:
    path = os.path.join(target_dir, f)
    try:
        with Image.open(path) as img:
            if img.size[0] != main_width:
                img.close()
                os.remove(path)
                deleted_count += 1
    except:
        pass # Déjà géré au dessus

print(f"\nNettoyage terminé :")
print(f"- {len(corrupted_files)} fichiers corrompus supprimés")
print(f"- {deleted_count} fichiers de mauvaise taille supprimés")

Analyse des fichiers dans : /content/spectros_local/3s_prediction_mel_spectrograms/3s_prediction_mel_spectrograms
Utilisation du sous-dossier detecte : /content/spectros_local/3s_prediction_mel_spectrograms/3s_prediction_mel_spectrograms/3s_prediction_mel_spectrograms
Scan des dimensions et intégrité...


100%|██████████| 200117/200117 [02:33<00:00, 1307.29it/s]



⚠️ 1 fichier(s) corrompu(s) détecté(s). Suppression...

Largeur standard identifiée : 522px

Nettoyage terminé :
- 1 fichiers corrompus supprimés
- 0 fichiers de mauvaise taille supprimés


In [7]:
import csv
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

# --- CONFIGURATION DU CALCUL ---
BATCH_SIZE = 128
NUM_WORKERS = 2

# --- DÉTECTION AUTOMATIQUE PROFONDE ---
import glob
# On cherche n'importe quel fichier PNG dans l'arborescence complète
found_files = glob.glob(os.path.join(path_imgs, "**/*.png"), recursive=True)

if not found_files:
    raise FileNotFoundError(f" Aucune image trouvée dans {path_imgs}. Vérifie l'extraction ZIP.")

# On définit le dossier cible comme étant celui qui contient les images
target_dir = os.path.dirname(found_files[0])
print(f" Images localisées dans : {target_dir}")
print(f" Total d'images trouvées : {len(found_files)}")

checkpoint_csv = path_csv_final

# Gestion automatique du sous-dossier apres unzip
if os.path.exists(target_dir):
    content = [f for f in os.listdir(target_dir) if not f.startswith('.')]
    if len(content) == 1 and os.path.isdir(os.path.join(target_dir, content[0])):
        target_dir = os.path.join(target_dir, content[0])
        print(f"Utilisation du sous-dossier : {target_dir}")

# 1. Preparation de la reprise
processed_files = set()
if os.path.exists(checkpoint_csv):
    try:
        existing_df = pd.read_csv(checkpoint_csv, usecols=["filename"])
        processed_files = set(existing_df["filename"].tolist())
        print(f"Reprise detectee : {len(processed_files)} images deja traitees.")
    except Exception as e:
        print(f"Creation d'un nouveau fichier (Fichier actuel illisible ou absent)")


# 2. Filtrage du Dataset (Version filtrée par animal)
class FilteredSpectroDataset(Dataset):
    def __init__(self, img_dir, processed_set, animal_filter, transform=None):
        self.img_dir = img_dir
        if not os.path.exists(img_dir):
            raise FileNotFoundError(f"Dossier introuvable : {img_dir}")

        # On liste tous les fichiers PNG
        all_f = sorted([f for f in os.listdir(img_dir) if f.endswith('.png')])

        # DOUBLE FILTRE :
        # 1. Le fichier n'est pas dans processed_set (reprise sur erreur)
        # 2. Le nom du fichier contient le nom de l'animal choisi
        self.img_names = [
            f for f in all_f
            if f not in processed_set and animal_filter in f
        ]

        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.img_names[idx]

# Initialisation du dataset avec le filtre défini en Cellule 2
dataset_todo = FilteredSpectroDataset(target_dir, processed_files, ANIMAL_TO_PROCESS, transform=transform)
dataloader = DataLoader(dataset_todo, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Nombre d'images de {ANIMAL_TO_PROCESS} restant à traiter : {len(dataset_todo)}")

# 3. Boucle d'extraction
model.to(device)
model.eval()

# Vérification si le fichier existe et n'est pas vide pour décider de l'écriture du header
file_exists = os.path.isfile(checkpoint_csv) and os.path.getsize(checkpoint_csv) > 0

with open(checkpoint_csv, mode='a', newline='') as f:
    writer = csv.writer(f)
    header_done = file_exists

    with torch.no_grad():
        for i, (batch_imgs, batch_names) in enumerate(tqdm(dataloader, desc="Extraction")):
            batch_imgs = batch_imgs.to(device)
            if device.type == 'cuda':
                batch_imgs = batch_imgs.half()

            # Inference
            outputs = model.forward_features(batch_imgs)
            embeddings = outputs["x_norm_clstoken"].cpu().float().numpy()

            # Ecriture
            for j in range(len(batch_names)):
                if not header_done:
                    # Reconstruction dynamique du header avec les dimensions réelles
                    header = ["filename"] + [f"dim_{k}" for k in range(embeddings.shape[1])]
                    writer.writerow(header)
                    header_done = True

                row = [batch_names[j]] + embeddings[j].tolist()
                writer.writerow(row)

            # Securite : Sauvegarde physique tous les 10 batches
            if (i + 1) % 10 == 0:
                f.flush()
                os.fsync(f.fileno())

print(f"Extraction terminee. Fichier enregistre : {checkpoint_csv}")

# --- SÉCURITÉ TRANSFERT GOOGLE DRIVE ---

try:
    print("Synchronisation des données vers Google Drive...")
    # On force l'écriture des buffers et la synchronisation physique
    from google.colab import drive
    drive.flush_and_unmount()
    print("Synchronisation réussie. Drive démonté.")
except Exception as e:
    print(f"Erreur lors de la synchronisation : {e}")

# --- DÉCONNEXION AUTOMATIQUE ---

import time
# On laisse une petite marge de 5 secondes par précaution
time.sleep(5)

print("Fermeture de la session Colab...")
from google.colab import runtime
runtime.unassign()


 Images localisées dans : /content/spectros_local/3s_prediction_mel_spectrograms/3s_prediction_mel_spectrograms/3s_prediction_mel_spectrograms
 Total d'images trouvées : 200116
Nombre d'images de HiP417 restant à traiter : 120075


Extraction: 100%|██████████| 939/939 [2:17:13<00:00,  8.77s/it]


Extraction terminee. Fichier enregistre : /content/drive/MyDrive/audio_classification_sam/embeddings/3s_prediction_mel_spectrograms_HiP417.csv
Synchronisation des données vers Google Drive...
Synchronisation réussie. Drive démonté.
Fermeture de la session Colab...
